In [1]:
import torch 
import torch.nn as nn

In [19]:
class ResNetBlock(nn.Module):
    def __init__(self, channels: int):
        # Khởi tạo khối ResNet với channels: là số chiều của ảnh đầu vào
        super(ResNetBlock, self).__init__()
        self.channels = channels
        self.num_groups = 8
        # Khởi tạo cho khối thứ 1
        self.norm1 = nn.GroupNorm(num_groups=self.num_groups, num_channels=self.channels)
        self.act1 = nn.SiLU()

        self.conv1 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3,
                               stride=1, padding=1)
        # Khởi tạo cho khối thứ 2
        self.norm2 = nn.GroupNorm(num_groups=self.num_groups, num_channels=self.channels)
        self.act2 = nn.SiLU()

        self.conv2 = nn.Conv2d(in_channels=channels, out_channels=channels, kernel_size=3, 
                               stride=1, padding=1)

        
    def forward(self, x):
        residual = x
        # Khối 1
        x = self.norm1(x)
        x = self.act1(x)
        x = self.conv1(x)
        # Khối 2
        x = self.norm2(x)
        x = self.act2(x)
        x = self.conv2(x)

        return x + residual

In [21]:
# Cài đặt Encoder
class Encoder(nn.Module):
    def __init__(self, latent_features: int):
        super(Encoder, self).__init__()
        self.latent_features = latent_features
        self.conv0 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.resnet1 = ResNetBlock(channels=32)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=2, padding=1)
        self.resnet2 = ResNetBlock(channels=64)

        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=2, padding=1)
        self.resnet3 = ResNetBlock(channels=128)

        self.conv6 = nn.Conv2d(in_channels=128, out_channels= self.latent_features * 2, kernel_size=3, stride=2, padding = 1)
        
    def forward(self, x):
        x0 = self.conv0(x)
        x1 = self.resnet1(x0)

        x2 = self.conv2(x1)
        x3 = self.resnet2(x2)

        x4 = self.conv4(x3)
        x5 = self.resnet3(x4)
        
        x6 = self.conv6(x5)
        # Sử dụng torch.chunk để chia đôi tensor.
        muy, log_var = torch.chunk(x6, chunks=2, dim= 1)
        # Giải thích: mu và log_var sẽ có shape là (B, latent_features, 32, 32)
        return muy, log_var

In [5]:
# Latent space and reparameterization trick
def reparameterization_trick(mu: torch.Tensor, log_var: torch.Tensor):
    # 1. Lấy độ lệch chuẩn
    stdev = torch.exp(0.5 * log_var)

    # 2. Lấy mẫu nhiễu epsilon từ phân phối chuẩn tắc N(0, 1)
    # Hàm torch.randn_like cực kỳ tiện lợi: nó tự động tạo ra một tensor chứa nhiễu
    # có cùng kích thước (shape), cùng kiểu dữ liệu (dtype) và 
    # nằm trên cùng thiết bị (CPU hay GPU) với tensor `mu`.
    epsilon = torch.randn_like(mu)

    z = mu + stdev * epsilon

    return z

In [17]:
# Decoder
class Decoder(nn.Module):
    def __init__(self, latent_features: int):
        super(Decoder, self).__init__()
        self.latent_features = latent_features
        self.sigmoid = nn.Sigmoid()
        self.conv0 = nn.ConvTranspose2d(in_channels=self.latent_features, out_channels=128, kernel_size=4, stride=2,
                                        padding=1)
        self.conv1 = nn.ConvTranspose2d(in_channels=128, out_channels=128, kernel_size=4, stride=1,
                                        padding=1)
        self.conv2 = nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=4, stride=2,
                                        padding=1)
        self.conv3 = nn.ConvTranspose2d(in_channels=64, out_channels=64, kernel_size=4, stride=1,
                                        padding=1)
        self.conv4 = nn.ConvTranspose2d(in_channels=64, out_channels=32, kernel_size=4, stride=2,
                                        padding=1)
        self.conv5 = nn.ConvTranspose2d(in_channels=32, out_channels=32, kernel_size=4, stride=1, 
                                        padding=1)
        self.conv6 = nn.ConvTranspose2d(in_channels=32, out_channels=1, kernel_size=3, stride=1, 
                                        padding=1)
        
        self.resnet1 = ResNetBlock(channels=128, conv=self.conv1)
        self.resnet2 = ResNetBlock(channels=64, conv=self.conv3)
        self.resnet3 = ResNetBlock(channels=32, conv=self.conv5)
    
    def forward(self, x):
        x0 = self.conv0(x)
        x1 = self.resnet1(x0)
        x2 = self.conv2(x1)
        x3 = self.resnet2(x2)
        x4 = self.conv4(x3)
        x5 = self.resnet3(x4)
        logits = self.conv6(x5) # Shape: (B, 3, 256, 256)
        output = self.sigmoid(logits) # Ép giá trị về [0, 1]

        return output

In [7]:
# VAE class
class VAE(nn.Module):
    def __init__(self, latent_features: int):
        super(VAE, self).__init__()
        # Khởi tạo encoder và decoder
        self.encoder = Encoder(latent_features=latent_features)
        self.decoder = Decoder(latent_features=latent_features)
    
    def forward(self, x):
        # Encoder
        mu, log_var = self.encoder(x)
        # Reparameterization trick
        z = reparameterization_trick(mu=mu, log_var=log_var)
        # Decoder
        output = self.decoder(z)
        return output, mu, log_var

In [10]:
import torch.nn.functional as F

In [8]:
import torch.nn.functional as F
# Hàm loss
def vae_loss_fn(model, batch, lambda_rec = 1.0, lambda_kl = 1.0):
    # Chạy model
    output, mu, log_var = model(batch)
    # Reconstruction loss: Mean Absolute Loss
    reconstruction_loss = F.l1_loss(output, batch)
    # KL Loss
    kl_loss = 0.5 * torch.sum(mu.pow(2) + log_var.exp() - log_var - 1, dim=(1, 2, 3))
    kl_loss = torch.mean(kl_loss) # Lấy trung bình trên toàn batch
    # Tổng hợp Loss
    total_loss = lambda_rec * reconstruction_loss + lambda_kl * kl_loss
    
    return total_loss

In [9]:
# Train_step
def train_step(optimizer, batch, model, lambda_rec, lambda_kl):
    x = batch['image']
    # Xóa đạo hàm
    optimizer.zero_grad()
    # Forward pass
    total_loss = vae_loss_fn(model=model, batch=x, lambda_rec=lambda_rec, lambda_kl=lambda_kl)

    # Tình đạo hàm
    total_loss.backward()
    # Cập nhật trọng số
    optimizer.step()

    return total_loss.item()

In [11]:
import torch.optim as optim
from tqdm import tqdm

In [12]:

# Vòng lặp huấn luyện
def run_training(model, data_loader, config, device):
    model.train()
    model.to(device)

    # Khởi tạo optimizer
    optimizer = optim.AdamW(model.parameters(), lr=config['learning_rate'])

    # Vòng lặp epoch
    for epoch_idx in range(config['num_epochs']):
        epoch_losses = []

        # tqdm để tạo thanh tiến trình 
        progess_bar = tqdm(enumerate(data_loader), total=len(data_loader), desc=f"Epoch {epoch_idx+1}/{config['num_epochs']}")

        # Vòng lặp batch
        for i, batch in progess_bar:
            x = batch['image'].to(device)
            # Xóa sạch đạo hàm
            optimizer.zero_grad()
            # Tính loss
            loss = vae_loss_fn(
                model=model,
                batch=x,
                lambda_rec=config['lambda_rec'],
                lambda_kl=config['lambda_kl']
            )
            # Backpropagation
            loss.backward()

            # Cập nhật trọng số
            optimizer.step()

            current_loss = loss.item()
            epoch_losses.append(current_loss)

            # Cập nhật thanh tiến trình
            progess_bar.set_postfix(loss=f"{current_loss:.4f}")
        
        # Tính trung bình loss sau mỗi epoch
        avg_epoch_loss = sum(epoch_losses) / len(epoch_losses)
        print(f"Epoch {epoch_idx+1} | Loss trung bình: {avg_epoch_loss:.4f}\n")


    print("Huấn luyện xong")
    return model


In [13]:
import yaml
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset

with open("config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

In [ ]:
# Xây dựng Wrapper cho Dataset
class MNISTDataset(Dataset):
    def __init__(self, root="./data", train=True):
        # Đổi kích thước lên 32*32 và chuyển thành Tensor [0, 1]
        self.transform = transforms.Compose([
            transforms.Resize(32),
            transforms.ToTensor()
        ])
        self.dataset = datasets.MNIST(root=root, train=train, download=True, transform=self.transform)

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, index):
        img, label = self.dataset[index]
        return {'image': img, 'label': label}

# Khởi tạo Dataset và Dataloader
train_dataset = MNISTDataset(train=True)
train_loader = DataLoader(train_dataset, batch_size=config['batch_size'])

100%|██████████| 9.91M/9.91M [00:12<00:00, 776kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 103kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 801kB/s]
100%|██████████| 4.54k/4.54k [00:00<?, ?B/s]


Device:  cuda


In [18]:
# Khởi tạo device
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Device: ", device)

# Model
model = VAE(latent_features=config['latent_features'])

# Huấn luyện
trained_model = run_training(
    model=model,
    data_loader=train_loader,
    config=config,
    device=device
)

Device:  cuda


Epoch 1/10:   0%|          | 0/469 [00:01<?, ?it/s]


RuntimeError: The size of tensor a (10) must match the size of tensor b (8) at non-singleton dimension 3